<a href="https://colab.research.google.com/github/Garcchi-Prog/Lab1EDDII/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Imports ##

from typing import Optional, Tuple
import csv

In [2]:
## Clase Nodo ##

class Node:
    def __init__(self, data: list) -> None:
        self.data = data
        self.id = data[0]
        self.title = data[1]
        
        # Convertir datos del CSV a números para el cálculo [cite: 11-24]
        try:
            rating = float(data[3])
            num_reviews = float(data[4])
            pos = float(data[11])
            neg = float(data[12])
            neu = float(data[13])
            self.satis = self.calcular_satisfaccion(rating, pos, neg, neu, num_reviews)
        except (ValueError, IndexError):
            self.satis = 0.0

        self.izquierda: Optional["Node"] = None
        self.derecha: Optional["Node"] = None
        self.altura = 1
    def calcular_satisfaccion(self, rating, pos, neg, neu, num_reviews):
        # Fórmula según requerimiento [cite: 8-10] con precisión de 5 decimales [cite: 7]
        if num_reviews > 0:
            val = (rating * 0.7) + (((5 * pos) + (3 * neu) + neg) / num_reviews) * 0.3
            return round(val, 5)
        return 0.0

In [3]:
## Clase Arbol ##

class Tree:
    def __init__(self) -> None:
        self.raiz = None

    def insertar(self, datos_fila):
        self.raiz = self._insertar_recursivo(self.raiz, datos_fila)

    def _insertar_recursivo(self, nodo, datos):
        if not nodo:
            return Node(datos)
        
        nuevo_temp = Node(datos)
        if nuevo_temp.satis < nodo.satis:
            nodo.izquierda = self._insertar_recursivo(nodo.izquierda, datos)
        elif nuevo_temp.satis > nodo.satis:
            nodo.derecha = self._insertar_recursivo(nodo.derecha, datos)
        else:
            return nodo # No permite duplicados exactos de satisfacción [cite: 6]

        return equilibrar(nodo)

    def buscar_por_id(self, id_buscado, nodo):
        if not nodo:
            return None
        if nodo.id == id_buscado:
            return nodo
        
        izq = self.buscar_por_id(id_buscado, nodo.izquierda)
        if izq: return izq
        return self.buscar_por_id(id_buscado, nodo.derecha)

    def delete_node(self, dataSearch, dataset, option: int):
        current = self.raiz
        parent = None
        searchedNode = self.search_node(dataSearch, dataset, option)

        if type(searchedNode) != Node:
            print(searchedNode)
        else: 
            pass

In [4]:
## Codigo AVL ## 

def rotacion_doble_izquierda_derecha(nodo):
    #Se hace una rotacion simple a la izquierda del hijo izquierdo del nodo desbalanceado
    nodo.izquierda = rotacion_simple_izquierda(nodo.izquierda)
    #Luego se hace una rotacion simple a la derecha del nodo desbalanceado
    n_raiz= rotacion_simple_derecha(nodo)
    return n_raiz

def rotacion_doble_derecha_izquierda(nodo):
    #Se hace una rotacion simple a la derecha del hijo derecho del nodo desbalanceado
    nodo.derecha = rotacion_simple_derecha(nodo.derecha)
    #Luego se hace una rotacion simple a la izquierda del nodo desbalanceado
    n_raiz= rotacion_simple_izquierda(nodo)
    return n_raiz


def altura(nodo):
    if not nodo:
        return 0
    return nodo.altura

def obtener_equilibrio(nodo):
    if not nodo:
        return 0
    return altura(nodo.izquierda) - altura(nodo.derecha)

def rotacion_simple_derecha(y):
    x = y.izquierda
    T2 = x.derecha
    x.derecha = y
    y.izquierda = T2
    y.altura = 1 + max(altura(y.izquierda), altura(y.derecha))
    x.altura = 1 + max(altura(x.izquierda), altura(x.derecha))
    return x

def rotacion_simple_izquierda(x):
    y = x.derecha
    T2 = y.izquierda
    y.izquierda = x
    x.derecha = T2
    x.altura = 1 + max(altura(x.izquierda), altura(x.derecha))
    y.altura = 1 + max(altura(y.izquierda), altura(y.derecha))
    return y

def equilibrar(nodo):
    if not nodo:
        return nodo
    
    # Actualizar altura
    nodo.altura = 1 + max(altura(nodo.izquierda), altura(nodo.derecha))
    
    balance = obtener_equilibrio(nodo)

    # Caso Izquierda-Izquierda
    if balance > 1 and obtener_equilibrio(nodo.izquierda) >= 0:
        return rotacion_simple_derecha(nodo)
    # Caso Izquierda-Derecha
    if balance > 1 and obtener_equilibrio(nodo.izquierda) < 0:
        nodo.izquierda = rotacion_simple_izquierda(nodo.izquierda)
        return rotacion_simple_derecha(nodo)
    # Caso Derecha-Derecha
    if balance < -1 and obtener_equilibrio(nodo.derecha) <= 0:
        return rotacion_simple_izquierda(nodo)
    # Caso Derecha-Izquierda
    if balance < -1 and obtener_equilibrio(nodo.derecha) > 0:
        nodo.derecha = rotacion_simple_derecha(nodo.derecha)
        return rotacion_simple_izquierda(nodo)
    
    return nodo

In [5]:
#CODIGO DE LAS OPERACIONES PARA BUSCAR PADRE, ABUELO Y TIO (RECURSIVO)

def BuscarPadre(raiz, nodo):
    if raiz is None:
        return None
    
    # Verificamos si  la raiz es el padre del nodo buscado tanto en la izquierda como en la derecha
    if (raiz.izquierda is  nodo) or (raiz.derecha is     nodo):
        return raiz
    
    #Se aplica recursion  para buscar el padre en el subarbol izquierdo 
    padre_izquierda = BuscarPadre(raiz.izquierda, nodo)
    if padre_izquierda is not None:
        return padre_izquierda
    
    #Se aplica recursion para buscar el padre en el subarbol derecho
    padre_derecha = BuscarPadre(raiz.derecha, nodo)
    if padre_derecha is not None:
        return padre_derecha

    #Si no se encuentra el nodo en ninguno de los subarboles, se devuelve none
    return None

def BuscarAbuelo (raiz, nodo):
    if raiz is None:
        return None
    
    #Se busca el padre del nodo dado
    padre = BuscarPadre(raiz, nodo)
    if padre is None:
        return None
    
    #Se busca el abuelo utilizando la funcion BuscarPadre, solamente que el nodo a buscar sera el del padre 
    abuelo=BuscarPadre(raiz, padre)
    return abuelo

def BuscarTio (raiz, nodo):
    if raiz is None:
        return None

    #Se busca al padre para tener una referencia para buscar el abuelo
    padre=BuscarPadre(raiz, nodo)
    #Si el padre no existe, no se puede encontrar al tio
    if padre is None:
        return None
    
    #Se busca el abuelo para tener una referencia para localizar al tio
    abuelo=BuscarPadre(raiz, padre)
    #Si el abuelo no existe, no se puede encontrar al tio
    if abuelo is None:
        return None
    
    #Caso 1: El hijo izquierdo del abuelo es el padre, por lo tanto el hijo derecho sera el tio
    if abuelo.izquierda is padre:
        return abuelo.derecha
    #Caso 2: El hijo derecho del abuelo es el padre, por lo tanto el hijo izquierdo sera el tio
    elif abuelo.derecha is padre:
        return abuelo.izquierda
    #Caso 3: El padre no es hijo del abuelo, por lo cual se retorna None
    else:
        return None

In [6]:
#Codigo para obtener el nivel de un nodo

def obtener_nivel(raiz, nodo):
    if raiz is None:
        return -1

    if raiz is nodo:
        return 0

    nivel_izquierda = obtener_nivel(raiz.izquierda, nodo)
    if nivel_izquierda != -1:
        return nivel_izquierda + 1
        
    nivel_derecha = obtener_nivel(raiz.derecha, nodo)
    if nivel_derecha != -1:
        return nivel_derecha + 1


In [ ]:
def menu():
    mi_arbol = Tree()
    archivo = 'dataset_courses_with_reviews.csv'

    # Pre-indexar TODO el CSV por ID al inicio (O(1) lookup)
    indice_csv = {}
    try:
        with open(archivo, mode='r', encoding='utf-8') as f:
            lector = csv.reader(f)
            next(lector)
            for fila in lector:
                if fila:
                    indice_csv[fila[0]] = fila
        print(f">>> CSV indexado: {len(indice_csv)} cursos disponibles.")
    except FileNotFoundError:
        print(">>> ERROR: No se encontró el archivo CSV.")
        return

    # Cargar los primeros 50 usando el índice
    for i, fila in enumerate(indice_csv.values()):
        if i >= 50:
            break
        mi_arbol.insertar(fila)
    print(">>> Árbol listo con 50 registros.")

    while True:
        print("\n--- LABORATORIO 1: UDEMY AVL ---")
        print("1. Insertar curso manualmente (ID)")
        print("2. Buscar curso por ID")
        print("3. Salir")

        opc = input("Seleccione: ")

        if opc == "1":
            id_input = input("Ingrese ID del curso en el CSV: ")
            fila = indice_csv.get(id_input)  # O(1) en vez de recorrer 83k filas
            if fila:
                mi_arbol.insertar(fila)
                print(f"Curso '{fila[1]}' insertado correctamente.")
            else:
                print("ID no existe en el dataset.")

        elif opc == "2":
            id_busc = input("ID a buscar: ")
            res = mi_arbol.buscar_por_id(id_busc, mi_arbol.raiz)
            if res:
                print(f"ENCONTRADO: {res.title}")
                print(f"Satisfacción: {res.satis}")
            else:
                print("No se encontró el curso en el árbol.")

        elif opc == "3":
            break

if __name__ == "__main__":
    menu()

>>> CSV indexado: 3 cursos disponibles.
>>> Árbol listo con 3 registros.

--- LABORATORIO 1: UDEMY AVL ---
1. Insertar curso manualmente (ID)
2. Buscar curso por ID
3. Salir
